# The warehouse against its sources

Step 5 built the warehouse: ODS from the raw files, DWD with every version and its `known_at`, DWS through `as_of`, ADS with labels and outcomes. This notebook asks whether what was built is true to the sources, from two sides. The reasons behind each rule are in [notebook 04](04_ods_to_dwd.ipynb); here the built models are checked, read from dbt's datasets in dev.

| Question | Where |
| --- | --- |
| Is every version's `known_at` consistent with the fetches that brought it, for every source? | Section 1 |
| Does one known day, traced by hand from the raw files, come out right in every layer? | Section 2 (next) |

## 0. Setup

**Step.** `sql()` runs a query against dbt's DWD dataset in dev and prints what it scanned, as in notebooks 04 and 05. `ODS` is the parsers' dataset, read here only for its manifest, `raw_files`: one row per raw file and ODS table, with the fetch time of each file (`fetched_at`) and how it was fetched (`load_type`: `daily`, `initial` or `recapture`).

In [1]:
import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.width", 160, "display.max_columns", 30, "display.max_colwidth", 60)


def setting(name: str) -> str:
    if value := os.environ.get(name):
        return value
    lines = Path("../.env").read_text().splitlines()
    env = dict(line.split("=", 1) for line in lines if "=" in line and not line.startswith("#"))
    return env[name]


PROJECT = setting("JTF_BIGQUERY_PROJECT")
DWD = f"{PROJECT}.{setting('JTF_DBT_DATASET_PREFIX')}_dwd"
ODS = f"{PROJECT}.ods"
bq = bigquery.Client(project=PROJECT)


def sql(query: str) -> pd.DataFrame:
    """Run a query against dbt's DWD dataset in dev; print what it scanned and how long it took."""
    config = bigquery.QueryJobConfig(default_dataset=DWD, use_query_cache=False)
    job = bq.query(query, job_config=config)
    rows = job.to_dataframe(create_bqstorage_client=False)
    seconds = (job.ended - job.started).total_seconds()
    print(f"[{(job.total_bytes_processed or 0) / 1e6:.1f} MB scanned, {seconds:.0f} s in BigQuery]")
    return rows


print(f"DWD dataset: {DWD} | ODS: {ODS}")

DWD dataset: jepx-tokyo-forecast-dev.dev_dwd | ODS: jepx-tokyo-forecast-dev.ods


## 1. `known_at` per source

What "right" means differs by source, because sources reach us differently ([notebook 04](04_ods_to_dwd.ipynb), section 1). A version is dated one of two ways, its `basis`:

- **`rule`**: a value we hold only from history, fetched after the fact, is dated by the source's history rule: when the source published it, as notebook 04 worked out per source (sections 2, 3.1, 3.2 and 6)
- **`captured`**: a value first seen by a fetch is known at that fetch, never earlier

The plan names three checks to run on real data after every load (Testing, Warehouse tests): `rule` versions sit before their source's capture start, `captured` versions carry the fetch time of a file holding the value, and no `known_at` lies after the newest fetch. Before turning them into tests for every source, this section asks what each source has been fetched as, and whether the checks hold on the models as built.

### 1.1 How each source reached us

**Step.** Per source in the manifest, how many files came by each kind of fetch, and when the first and last came. A source's *capture start* is its first `daily` fetch (notebook 04, section 1): before it, history is all we have, and the rule dates it.

In [2]:
fetches = sql(f"""
SELECT source, load_type, COUNT(DISTINCT path) AS files,
  MIN(fetched_at) AS first_fetch, MAX(fetched_at) AS last_fetch
FROM `{ODS}.raw_files`
WHERE source IN ('tepco_actuals', 'meti_fit_capacity', 'openmeteo_previous_runs',
                 'openmeteo_single_runs', 'openmeteo_live')
GROUP BY source, load_type
ORDER BY source, load_type
""")
fetches

[0.2 MB scanned, 0 s in BigQuery]


,source,load_type,files,first_fetch,last_fetch
0,meti_fit_capacity,daily,3,2026-11-16 23:45:19.719609+00:00,2026-11-17 23:45:18.096105+00:00
1,meti_fit_capacity,initial,119,2026-11-16 00:40:24.962795+00:00,2026-11-16 00:45:32.972052+00:00
2,openmeteo_live,daily,5,2026-11-15 02:36:27.058501+00:00,2026-11-17 23:45:18.697832+00:00
3,openmeteo_previous_runs,initial,36,2026-11-15 04:52:42.808649+00:00,2026-11-16 12:29:54.176374+00:00
4,openmeteo_single_runs,initial,1362,2026-11-15 05:01:46.615911+00:00,2026-11-16 12:46:48.318503+00:00
5,tepco_actuals,daily,12,2026-11-15 02:04:16.643393+00:00,2026-11-17 23:45:20.405456+00:00
6,tepco_actuals,initial,41,2026-11-15 04:52:40.111927+00:00,2026-11-15 04:54:24.243215+00:00
7,tepco_actuals,recapture,1,2026-11-15 05:02:48.862189+00:00,2026-11-15 05:02:48.862189+00:00


**What it shows.** **Observed:**

- **The actuals and capacity have a daily capture, so a capture start.** The actuals' first daily fetch came at 11:04 JST on 15 November, before their history was loaded at 13:52: the history files fetched after it are known at their fetch, the late backfill of [notebook 04](04_ods_to_dwd.ipynb), section 2.2. Capacity's first daily fetch came at 08:45 on 17 November, after its 119 history files the morning before
- **Previous Runs and Single Runs came only by `initial` fetches**, 36 and 1,362 files on 15 and 16 November. The capture jobs never fetch them (**Source**: [ingest README](../jtf/src/jtf/ingest/README.md): *"history-only sources (`openmeteo_previous_runs`, `openmeteo_single_runs`) have no daily schedule"*), so they have no capture start
- **The live weather came only by daily fetches**, five files since a test run on 15 November

**Learned → design:** "rule versions before capture start" has nothing to compare against for the weather history, the source with the most rule-dated versions. The rule needs a check that every source has: whatever a rule says, it can't date a value later than the fetch that first brought it, since nothing is published after we fetched it. That is check D below.

### 1.2 Five checks over every versioned model

**Step.** Every DWD model others read through `as_of` carries `known_at` and `basis` (the rules' layer test makes sure of the first). Each model is checked whole, against the manifest, by five counts of versions that break a check:

| Check | Counts the versions where |
| --- | --- |
| **A**: captured, never early | `basis = 'captured'` and `known_at` lies before the fetch that held the value |
| **B**: captured, exactly | `basis = 'captured'` and `known_at` differs from that fetch |
| **C**: rule, before capture start | `basis = 'rule'` and `known_at` lies at or after the source's capture start, where it has one |
| **D**: rule, never after its fetch | `basis = 'rule'` and `known_at` lies after the fetch that first held the value |
| **E**: nothing after the newest fetch | `known_at` lies after the newest fetch of the model's sources |

The fetch that held a value is each model's own column: `first_seen` for the actuals, the capacity figures and the weather history, `first_held` for the capacity releases (when both of a release's files were first held), `captured_at` for the live weather. `dwd_weather_slots` has none: it is made from the hours of the three weather models, so only E applies to it. A check that doesn't apply shows as empty.

In [3]:
MODELS = {
    # model: (its sources, the column with the fetch that held a value, its capture start's source)
    "dwd_tepco_actuals": (["tepco_actuals"], "first_seen", "tepco_actuals"),
    "dwd_capacity_releases": (["meti_fit_capacity"], "first_held", "meti_fit_capacity"),
    "dwd_capacity_area": (["meti_fit_capacity"], "first_seen", "meti_fit_capacity"),
    "dwd_weather_previous_runs": (["openmeteo_previous_runs"], "first_seen", None),
    "dwd_weather_single_runs": (["openmeteo_single_runs"], "first_seen", None),
    "dwd_weather_live": (["openmeteo_live"], "captured_at", "openmeteo_live"),
    "dwd_weather_slots": (
        ["openmeteo_previous_runs", "openmeteo_single_runs", "openmeteo_live"],
        None,
        None,
    ),
}


def counts(model: str, sources: list[str], fetch: str | None, start: str | None) -> str:
    def count(condition, applies=True):
        return f"COUNTIF({condition})" if applies else "CAST(NULL AS INT64)"

    listed = "', '".join(sources)
    capture_start = f"(SELECT capture_start FROM starts WHERE source = '{start}')"
    newest = f"(SELECT MAX(newest) FROM starts WHERE source IN ('{listed}'))"
    return f"""
SELECT '{model}' AS model, COUNT(*) AS versions, COUNTIF(basis = 'captured') AS captured,
  {count(f"basis = 'captured' AND known_at < {fetch}", bool(fetch))} AS a_captured_early,
  {count(f"basis = 'captured' AND known_at != {fetch}", bool(fetch))} AS b_captured_not_exact,
  {count(f"basis = 'rule' AND known_at >= {capture_start}", bool(start))} AS c_rule_after_start,
  {count(f"basis = 'rule' AND known_at > {fetch}", bool(fetch))} AS d_rule_after_fetch,
  {count(f"known_at > {newest}")} AS e_after_newest
FROM `{DWD}.{model}`"""


checks = sql(
    f"""WITH starts AS (
  SELECT source, MIN(IF(load_type = 'daily', fetched_at, NULL)) AS capture_start,
    MAX(fetched_at) AS newest
  FROM `{ODS}.raw_files` GROUP BY source
)"""
    + "\nUNION ALL".join(counts(model, *spec) for model, spec in MODELS.items())
)
checks.set_index("model")

[168.6 MB scanned, 0 s in BigQuery]


,versions,captured,a_captured_early,b_captured_not_exact,c_rule_after_start,d_rule_after_fetch,e_after_newest
model,,,,,,,
dwd_tepco_actuals,147075,423,0,0,0,0,0
dwd_capacity_releases,33,0,0,0,0,0,0
dwd_capacity_area,297,0,0,0,0,0,0
dwd_weather_previous_runs,2019029,104,0,0,<NA>,0,0
dwd_weather_single_runs,1274418,0,0,0,<NA>,0,0
dwd_weather_live,9630,9630,0,0,0,0,0
dwd_weather_slots,6606232,19546,<NA>,<NA>,<NA>,<NA>,0


**What it shows.** **Observed**, over 10.1 M versions in the seven models:

- **No version breaks a check that applies to it**
- **Every captured version is known exactly at its fetch** (check B): the actuals' 423, Previous Runs' 104 corrections, all 9,630 live versions
- **Three models have no captured version yet:** the capacity releases and their figures, since every release so far is dated by the rule (the first captured one is due by mid-December), and Single Runs, where no run has changed between fetches. On them, A and B have nothing to check

**Learned → design:** A, C, D and E become data tests after every load. B holds today but is a rule only where a model makes it one: the actuals and the capacity releases date a captured version at its fetch, and their tests already say so. The weather history dates a correction at its fetch or at the version it replaces, whichever is later (**Source**: [weather.sql](../warehouse/dbt/macros/weather.sql), `weather_versions`: *"A later version is known at the fetch that first held it, never before the version it replaces"*). A correction fetched before the previous version's rule time would be known after its fetch, by design. So for the weather the test is A: never before its fetch.

## The dbt tests this leads to

| Test | Check | On |
| --- | --- | --- |
| `<model>__rule_only_before_capture_start` (generic `rule_before_capture_start`) | C | The actuals, the capacity releases and their figures (`dwd_capacity_area`) |
| `<model>__rule_never_after_its_first_fetch` | D | The actuals, the capacity releases and figures, Previous Runs, Single Runs |
| `<model>__captured_never_before_its_fetch` | A | The capacity figures and Single Runs. The actuals and releases already test B, and Previous Runs tests A on its corrections |
| `<model>__no_known_at_after_newest_fetch` (generic `known_at_within_fetches`) | E | All seven |

The two generic tests read the ODS manifest ([tests/generic](../warehouse/dbt/tests/generic/)); the other two are expressions on the model's own fetch column. 17 tests in all, each seen to fail: the actuals built with their rule applied past capture start fail all three of theirs, and a violation planted in a copy of each other model's rows fails its test.

## What this exploration made known

| We assumed | We found | So |
| --- | --- | --- |
| The plan's three checks after every load apply to every source | Previous Runs and Single Runs came only by `initial` fetches: no daily capture, so no capture start for "rule before capture start" to compare against (section 1.1) | A rule is also checked against the fetch that first held each version: never dated after it. Every source gets that check; "before capture start" stays for the sources with a daily capture |
| A captured version carries the fetch time of a file holding the value (plan) | True, exactly, of every captured version today (section 1.2). The weather dates a correction no earlier than the version it replaces, so one could come later than its fetch | Exactly at its fetch where the model says so (the actuals, the capacity releases); elsewhere, never before it |
| `known_at` is right per source, as notebook 04 designed it | No version among 10.1 M in seven models breaks a check that applies to it (section 1.2) | The four checks become data tests after every load, 17 over the seven models |

### Still open

- **Section 2:** one known day, traced by hand from the raw files through every layer
- **The captured side of capacity and Single Runs has nothing to check yet.** It will once the first captured capacity release arrives (due by mid-December, progress, *Due on a date*) and once a run changes between fetches

## Where the sources and the data disagree

| Where | Says | We found | Handled by |
| --- | --- | --- | --- |
| [Phase-1 plan](../docs/phase1/README.md#ingestion-and-warehouse), Warehouse tests | After every load, `basis = rule` rows sit before their source's capture start | The weather history has no capture start (section 1.1) | The plan now also names the rule's check against the first fetch, for every source |
| The same paragraph | `basis = captured` rows carry the fetch time of a file holding the value | The weather dates a correction no earlier than the version it replaces, which can be later than its fetch (section 1.2) | Never before its fetch; exactly at it where the model says so |

## Sources

| Page | Supports |
| --- | --- |
| [Notebook 04](04_ods_to_dwd.ipynb), sections 1, 2.2, 3.1, 3.2 and 6 | Capture start per source, the late backfill, the weather's rules, capacity's rule |
| [weather.sql](../warehouse/dbt/macros/weather.sql), `weather_versions` | The weather history's versions: the first by rule, a later one at its fetch, never before the version it replaces |
| [ingest README](../jtf/src/jtf/ingest/README.md) | The history-only sources have no daily schedule |
| [Phase-1 plan](../docs/phase1/README.md#ingestion-and-warehouse), Warehouse tests | The checks after every load |